In [0]:
%pip install --upgrade langchain
%pip install --upgrade langchain-community
%pip install --upgrade langchain-text-splitters
%pip install --upgrade chromadb
%pip install --upgrade sentence-transformers
%pip install --upgrade langchain-google-genai
%pip install --upgrade pypdf

In [0]:
import os
from langchain_community.document_loaders import PyPDFDirectoryLoader  #loads all the pdfs
from langchain_text_splitters import RecursiveCharacterTextSplitter  #splits data aka the pdfs into smaller sections to find relative info quicker
from langchain_community.vectorstores import Chroma  #chroma is the storage where the splits and calcuatations is stored
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI  #basically connecting gemini to the code and converts text into a special code that the gemini ai can understand
from langchain_core.output_parsers import StrOutputParser  #converts back from gemini code into text for output
from langchain_core.runnables import RunnablePassthrough  #anything sent into the chain is seen as the input (user)
from langchain_core.prompts import ChatPromptTemplate  #the prompt sent into gemini (context (the specialised part of data that is relevant to the question) and input (the question itself))


os.environ['GOOGLE_API_KEY'] = '' #api key


loader = PyPDFDirectoryLoader("/Workspace/Users/kushal.vadukul2006@gmail.com/LangChain-RAG-AI/folder_of_cvs")
cvs = loader.load()
print("Loaded", len(cvs) ," Page's") #print number of pages of the cv's

splitter = RecursiveCharacterTextSplitter()
splits = splitter.split_documents(cvs) #splits the documents

convert_text2gemini = GoogleGenerativeAIEmbeddings(model = "models/gemini-embedding-001") 
store_chroma = Chroma.from_documents(splits, convert_text2gemini)
retrieve = store_chroma.as_retriever(search_kwargs={"k": 20})  #stores the info got upto to 20 chunks of text

llm = ChatGoogleGenerativeAI(model="gemini-3.8-flash", temperature=0) #type of gemini model, temp means no excess info printed

gemini_prompt = ChatPromptTemplate.from_template(
    "You are an expert HR recruitment assistant. Answer the user's question "
    "strictly and accurately using only the provided candidate resume context.\n\n"
    "{context}\n\n"
    "Question: {input}"
)

def format_docs(cvs):
    return "\n\n".join(doc.page_content for doc in cvs)     #retriever check chroma storage, grabs the text from chunk then gives list of texts, to make into actual sentence again for the prompt.

rag_chain = (
    {"context": retrieve | format_docs, "input": RunnablePassthrough()}  #passthrough anything that goes into chain is input
    | gemini_prompt
    | llm
    | StrOutputParser() #converts back from gemini code to text
)

user_input = input("Ask a question about the CVs: ")   #console input for user
response = rag_chain.invoke(user_input)

print("\n AI RESPONSE")
print(response)  #print response

with open("test_results.txt", "a") as f:  #writes the question and answer into a txt file in same folder called test_results.
    f.write(f"Question: {user_input}\n")
    f.write(f"Answer:\n{response}\n")
